# 00 - Cluster Spark

Aquí revisamos que el notebook sí esté conectado al cluster: el master, los executors, cómo se reparten las tareas y que HDFS funcione.

## 1. Sesión de Spark

No hace falta poner el master ni la memoria porque ya están en `spark-defaults.conf`.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("00_cluster")
    .getOrCreate()
)
sc = spark.sparkContext

print("Versión de Spark :", spark.version)
print("Master           :", sc.master)
print("Id de aplicación :", sc.applicationId)
print("UI de la app     :", sc.uiWebUrl)

Si `Master` empieza por `spark://` estamos en el cluster. Si sale `local[*]`, algo quedó mal y todo está corriendo solo en esta máquina.

## 2. Configuración

In [ ]:
import pandas as pd

claves = [
    "spark.master", "spark.driver.host", "spark.driver.memory",
    "spark.executor.cores", "spark.executor.memory",
    "spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
    "spark.serializer", "spark.sql.session.timeZone",
]
conf = dict(sc.getConf().getAll())
pd.DataFrame({"parámetro": claves, "valor": [conf.get(k, "(por defecto)") for k in claves]})

## 3. Executors

In [ ]:
import requests

url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/executors"
ejecutores = pd.DataFrame(requests.get(url, timeout=10).json())
ejecutores["memoria_GB"] = (ejecutores["maxMemory"] / 1024**3).round(2)
tabla = ejecutores[["id", "hostPort", "totalCores", "memoria_GB", "isActive"]]
tabla

In [ ]:
por_maquina = (
    tabla[tabla["id"] != "driver"]
    .assign(maquina=lambda d: d["hostPort"].str.split(":").str[0])
    .groupby("maquina")
    .agg(executors=("id", "count"), cores=("totalCores", "sum"), memoria_GB=("memoria_GB", "sum"))
)
print(f"Paralelismo por defecto: {sc.defaultParallelism} tareas simultáneas")
por_maquina

Tienen que aparecer las tres VMs.

## 4. Reparto de tareas

Lanzamos 200 tareas y cada una devuelve el nombre de la máquina donde corrió.

In [ ]:
import os, socket

def donde_corro(_):
    return f"{socket.gethostname()} ({os.environ.get('SPARK_LOCAL_IP', '?')})"

reparto = sc.parallelize(range(200), 200).map(donde_corro).countByValue()
pd.Series(dict(reparto), name="tareas").to_frame()

## 5. HDFS

Prueba de escritura y lectura: 10 millones de filas a Parquet y de vuelta.

In [ ]:
import time
from pyspark.sql import functions as F

ruta = "/tmp/prueba_00_cluster.parquet"   # ruta en HDFS

t0 = time.time()
(spark.range(10_000_000)
      .withColumn("grupo", F.col("id") % 5)
      .withColumn("valor", F.rand(seed=42))
      .write.mode("overwrite").parquet(ruta))
t_escritura = time.time() - t0

t0 = time.time()
resumen = spark.read.parquet(ruta).groupBy("grupo").agg(F.count("*").alias("filas"), F.avg("valor").alias("promedio"))
resultado = resumen.orderBy("grupo").toPandas()
t_lectura = time.time() - t0

print(f"Escritura: {t_escritura:.1f} s  |  Lectura + agregación: {t_lectura:.1f} s")
resultado

In [ ]:
# borrar la prueba
jvm = spark._jvm
fs = jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
print("Sistema de archivos por defecto:", fs.getUri())
fs.delete(jvm.org.apache.hadoop.fs.Path(ruta), True)

Cada grupo debería tener 2.000.000 de filas y un promedio cerca de 0,5, y el sistema de archivos tiene que ser `hdfs://`.

## Conclusiones

- El cluster es Spark standalone: VM1 tiene el master, un worker y el driver, y VM2 y VM3 tienen un worker cada una. HDFS está con replicación 2.
- La sesión quedó conectada al master, hay executors en las tres VMs, las tareas se reparten entre ellas y HDFS lee y escribe sin problema.